<a href="https://colab.research.google.com/github/huseyincenik/john_snow_labs/blob/main/generating_conll_files_from_pretrained_models/notebooks/prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prediction - Trained Model Inference

This notebook uses a trained custom NER model to make predictions on new texts.

**Google Drive Integration:**
- All files are saved to Google Drive
- Files are read from Google Drive
- Trained models are loaded from Google Drive
- All code is embedded in this notebook (no external Python files required)

## Steps:
1. **Google Drive Connection** - Mount Google Drive
2. **Setup & License** - Spark NLP Healthcare license and environment setup
3. **Model Loading** - Load trained custom NER model from Google Drive
4. **Prediction Pipeline** - Create prediction pipeline
5. **Make Predictions** - Run predictions on new texts
6. **Visualize Results** - Display and save prediction results

**Requirements:**
- `training.ipynb` notebook must be run first
- Trained model must exist at `models/trained/custom_ner_model` in Google Drive


## 1. Google Drive Connection


In [1]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Set project folder in Google Drive
PROJECT_FOLDER = '/content/drive/MyDrive/john_snow_labs_ner'
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Change working directory
os.chdir(PROJECT_FOLDER)

# Create folder structure
for folder in ['predictions']:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"✅ Project folder: {PROJECT_FOLDER}")
print(f"✅ Current directory: {os.getcwd()}")


Mounted at /content/drive
✅ Google Drive mounted
✅ Project folder: /content/drive/MyDrive/john_snow_labs_ner
✅ Current directory: /content/drive/MyDrive/john_snow_labs_ner


## 2. Setup & License Configuration


In [2]:
import json
import os

# Load license keys from Google Drive
license_path = f'{PROJECT_FOLDER}/spark_jsl.json'
if not os.path.exists(license_path):
    print("❌ License file not found!")
    print("Please upload spark_jsl.json to Google Drive at the project folder")
    print("You can upload it manually or use the following code:")
    print("from google.colab import files")
    print("uploaded = files.upload()")
    raise FileNotFoundError(f"License file not found at {license_path}")

with open(license_path) as f:
    license_keys = json.load(f)

# Set license keys as environment variables
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")


✅ License keys loaded
JSL Version: 6.2.1
Public Version: 6.2.0


In [3]:
# Install Java (required for Spark)
import subprocess

try:
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java is already installed: {java_version.split(chr(10))[0]}")

    if 'JAVA_HOME' not in os.environ:
        java_paths = [
            "/usr/lib/jvm/java-11-openjdk-amd64",
            "/usr/lib/jvm/java-8-openjdk-amd64",
            "/usr/lib/jvm/default-java"
        ]
        for path in java_paths:
            if os.path.exists(path):
                os.environ["JAVA_HOME"] = path
                print(f"✅ Set JAVA_HOME to: {path}")
                break
except Exception as e:
    print(f"Java check failed: {e}")
    print("Installing Java 11...")
    os.system('apt-get update -qq > /dev/null 2>&1')
    os.system('apt-get -y install -qq openjdk-11-jdk > /dev/null 2>&1')
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    print("✅ Java 11 installation attempted")

# Check GPU availability
# gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
# has_gpu = gpu_check.returncode == 0

# if has_gpu:
#     print("🚀 GPU detected! Installing PyTorch with CUDA support...")
#     %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# else:
#     print("Installing PyTorch (CPU version)...")
#     %pip install -q torch torchvision torchaudio

# Install PySpark and Spark NLP
%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Install Spark NLP Healthcare
%pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Install additional dependencies
%pip install -q pandas numpy

print("✅ All libraries installed successfully!")
# if has_gpu:
#     print("✅ GPU-accelerated PyTorch installed")


✅ Java is already installed: openjdk version "17.0.17" 2025-10-21
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 743.3/743.3 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 17.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.1 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.8/565.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.4/744.4 kB 16.0 MB/s eta 0:00:00
✅ All libraries installed successfully!


In [4]:
import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import MedicalNerModel, NerConverter
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.sql.types import StringType
from pyspark.sql import Row
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"🚀 GPU Detected: {gpu_name}")
    else:
        print("⚠️  No GPU detected. Using CPU mode.")
except ImportError:
    print("⚠️  PyTorch not available. GPU check skipped.")
    gpu_available = False

# Spark configuration
params = {
    "spark.driver.memory": "8G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.storage.cluster_tmp_dir": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled in Spark configuration")

# Start Spark session
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    spark.sparkContext.setLogLevel("ERROR")

    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized successfully")

except Exception as e:
    print(f"❌ Error starting Spark session: {e}")
    raise

spark


⚠️  No GPU detected. Using CPU mode.
Starting Spark session...
✅ Spark NLP Version: 6.2.2
✅ Spark NLP JSL Version: 6.2.1
✅ Spark session initialized successfully


In [5]:
# Load trained model from Google Drive
model_path = f"{PROJECT_FOLDER}/models/trained/custom_ner_model"

if not os.path.exists(model_path):
    print(f"❌ Model not found: {model_path}")
    print("Please run training.ipynb first to train the model.")
    raise FileNotFoundError(f"Model not found at {model_path}")

print(f"Loading trained model from {model_path}...")
custom_model = MedicalNerModel.load(model_path)
print("✅ Model loaded successfully")

# Load embeddings (required for the model)
print("Loading embeddings...")
embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
    .setInputCols(["sentence", "token"])\
    .setOutputCol("embeddings")
print("✅ Embeddings loaded")


Loading trained model from /content/drive/MyDrive/john_snow_labs_ner/models/trained/custom_ner_model...
✅ Model loaded successfully
Loading embeddings...
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
✅ Embeddings loaded


## 4. Create Prediction Pipeline


In [6]:

"""
NER Pipeline Module
Creates and executes Spark NLP Healthcare NER pipeline with multiple models
"""

import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import (
    MedicalNerModel,
    NerConverter
)
from sparknlp_jsl.annotator.ner.ner_converter_internal import NerConverterInternal

from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from typing import Optional, Dict, List
import warnings
warnings.filterwarnings('ignore')



document = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")

sentence = SentenceDetector()\
    .setInputCols(['document'])\
    .setOutputCol('sentence')

token = Tokenizer()\
    .setInputCols(['sentence'])\
    .setOutputCol('token')

converter = NerConverterInternal()\
    .setInputCols(["document", "token", "ner"])\
    .setOutputCol("ner_span")

ner_prediction_pipeline = Pipeline(stages=[
    document,
    sentence,
    token,
    embeddings,
    custom_model,
    converter])

empty_data = spark.createDataFrame([['']]).toDF("text")

prediction_model = ner_prediction_pipeline.fit(empty_data)

from sparknlp.base import LightPipeline

light_model = LightPipeline(prediction_model)


## 5. Prepare Input Texts


In [7]:
import os
from pyspark.sql import Row
from pyspark.sql.functions import col, explode
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import MedicalNerModel, NerConverter
from pyspark.ml import Pipeline

# ---------------------------
# Sample texts
# ---------------------------
sample_texts = [ "The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure due to hypertension.", "Dr. Smith recommended Metformin 500mg for type 2 diabetes mellitus and suggested regular HbA1c testing.", "The patient has a history of hypertension, hyperlipidemia, and is currently taking Lisinopril 10mg once daily, along with Atorvastatin 20mg at night.", "Patient presents with chest pain, shortness of breath, and palpitations. ECG shows ST elevation, and troponin levels are elevated.", "The medication dosage was increased to 20mg per day after consultation. The patient also started Omeprazole 40mg daily for gastroesophageal reflux disease.", "Administered Vancomycin 1g IV every 12 hours. Monitor renal function and complete blood count during therapy.", "Patient diagnosed with chronic kidney disease and is on Furosemide 40mg daily. Blood urea nitrogen and creatinine levels should be monitored.", "The patient received a flu vaccine and was advised to continue Vitamin D 2000 IU daily for bone health.", "Amoxicillin 500mg thrice daily was prescribed for bacterial pneumonia. Patient reported mild nausea as a side effect.", "The patient underwent MRI of the brain due to persistent headaches and dizziness.", "Patient presents with fever, chills, and productive cough. Chest X-ray confirms lobar pneumonia.", "The cardiologist prescribed Clopidogrel 75mg daily for post-stent thrombosis prevention.", "Patient has Type 1 Diabetes and is taking Insulin glargine 20 units at bedtime.", "The patient shows signs of anemia. Hemoglobin levels were 9.5 g/dL and iron supplements were recommended.", "Patient reports intermittent palpitations and shortness of breath. Echocardiogram shows mild mitral regurgitation.", "The patient started Hydroxychloroquine 200mg daily for rheumatoid arthritis treatment.", "Patient has a history of asthma and uses Albuterol inhaler 90 mcg as needed.", "The patient received a COVID-19 booster and is advised to continue monitoring oxygen saturation.", "Patient diagnosed with hypothyroidism and is taking Levothyroxine 50 mcg every morning.", "The patient presents with abdominal pain and diarrhea. Stool culture tested positive for Salmonella.", "Administered Ceftriaxone 2g IV once daily for severe bacterial infection.", "Patient has a history of hyperthyroidism and is on Methimazole 10mg daily.", "The patient reports insomnia and anxiety. Prescribed Diazepam 5mg at night.", "Patient was treated with Prednisone 20mg daily for acute exacerbation of COPD.", "Patient undergoing chemotherapy with Cisplatin 70mg/m2 every 3 weeks. Monitor renal function and complete blood count.", "The patient presents with rash, itching, and swelling. Prescribed Cetirizine 10mg daily." ]

text_df = spark.createDataFrame([Row(text=text) for text in sample_texts])
print(f"✅ Created DataFrame with {text_df.count()} texts")




✅ Created DataFrame with 26 texts


## 6. Run Predictions


In [9]:
predictions = light_model.transform(text_df)
predictions.select(
    "text",
    "ner_span.result"
).show(truncate=200)

exploded = predictions.select(
    col("text"),
    explode(col("ner_span.result")).alias("entity")
)
exploded.show(truncate=200)


+-----------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------+
|                                                                                                                                                       text|                                                                                       result|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------+
|                    The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure due to hypertension.|                             [Aspirin, pain management, monitor blood pressure, hyperte

## 7. NER Visualization with spark-nlp-display

We'll use **spark-nlp-display** to visualize NER results interactively.

### Key Features:
- Visual representation of named entities
- Color-coded entity labels
- HTML export for sharing results
- Customizable label colors and filters

In [14]:
from pyspark.sql import functions as F

ner_df = (
    predictions
    .select(
        F.col("text"),
        F.explode(
            F.arrays_zip(
                predictions.ner_span.result,
                predictions.ner_span.begin,
                predictions.ner_span.end,
                predictions.ner_span.metadata
            )
        ).alias("cols")
    )
    .select(
        F.col("text"),
        F.expr("cols['0']").alias("chunk"),
        F.expr("cols['1']").alias("begin"),
        F.expr("cols['2']").alias("end"),
        F.expr("cols['3']['entity']").alias("ner_label"),
        F.expr("cols['3']['sentence']").alias("sentence_id")
    )
    .filter("ner_label != 'O'")
)

ner_df.show(truncate=False)


+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------+-----+---+---------+-----------+
|text                                                                                                                                                       |chunk                   |begin|end|ner_label|sentence_id|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------+-----+---+---------+-----------+
|The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure due to hypertension.                    |Aspirin                 |27   |33 |DRUG     |0          |
|The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure due to hypertension.    

In [17]:
# Install spark-nlp-display if not already installed
!pip install spark-nlp-display -q

from sparknlp_display import NerVisualizer

# Initialize the visualizer
visualizer = NerVisualizer()

print("✅ NER Visualizer initialized")

✅ NER Visualizer initialized


In [18]:
for i, txt in enumerate(sample_texts):
    res = light_model.fullAnnotate(txt)[0]
    visualizer.display(
        res,
        label_col='ner_span',
        document_col='document',
        return_html=False
    )
